# 2. Storage Security

## Storage access methods — which one when?

| Method | Use case | Exam priority |
|--------|---------|---------------|
| **Entra ID + RBAC** | Applications with managed identity | ✅ Preferred |
| **Shared Access Signatures (SAS)** | Time-limited, scoped access for external parties | Common |
| **Access keys** | Full admin access (like root password) | ⚠️ Avoid |
| **Stored access policies** | Server-side SAS management (revoke without rotating keys) | Important |
| **Anonymous public access** | Static websites, public downloads | Disable by default |

In [ ]:
import json
from datetime import datetime, timedelta
import hashlib, hmac, base64

# Simulate SAS token generation (simplified)
def generate_sas_explanation(permissions: str, resource: str, expiry_hours: int, ip_range: str = '') -> dict:
    now = datetime.now()
    expiry = now + timedelta(hours=expiry_hours)
    sas_params = {
        'sv': '2023-11-03',        # service version
        'ss': 'b',                  # service (b=blob, f=file, q=queue, t=table)
        'srt': resource,            # resource type (s=service, c=container, o=object)
        'sp': permissions,           # permissions (r=read, w=write, d=delete, l=list)
        'se': expiry.strftime('%Y-%m-%dT%H:%M:%SZ'),  # expiry
        'st': now.strftime('%Y-%m-%dT%H:%M:%SZ'),     # start time
        'spr': 'https',             # protocol
    }
    if ip_range:
        sas_params['sip'] = ip_range
    
    return {
        'type': 'Account SAS' if resource != 'o' else 'Service SAS',
        'permissions': permissions,
        'expires': expiry.strftime('%Y-%m-%d %H:%M'),
        'ip_restriction': ip_range or 'none',
        'protocol': 'HTTPS only',
        'url_example': f'https://mysa.blob.core.windows.net/data/file.csv?{"&".join(f"{k}={v}" for k,v in sas_params.items())}&sig=<HMAC-SHA256>',
    }

print('=== SAS Token Types ===\n')

print('--- 1. Read-only blob access for 2 hours ---')
print(json.dumps(generate_sas_explanation('r', 'o', 2), indent=2))

print('\n--- 2. Read+write container access from specific IP ---')
print(json.dumps(generate_sas_explanation('rw', 'co', 24, '203.0.113.0/24'), indent=2))

print('\n--- 3. Full account SAS (dangerous!) ---')
print(json.dumps(generate_sas_explanation('rwdlac', 'sco', 720), indent=2))

print('\n⚠️ SAS tokens are signed by the account key. If the key is rotated, ALL SAS tokens become invalid.')
print('💡 Use stored access policies to revoke SAS tokens without rotating keys.')

## Storage protection features

| Feature | What it does | CLI |
|---------|-------------|-----|
| **Soft delete (blobs)** | Recover deleted blobs for N days | `az storage account blob-service-properties update --enable-delete-retention true --delete-retention-days 30` |
| **Soft delete (containers)** | Recover deleted containers | `--enable-container-delete-retention true --container-delete-retention-days 30` |
| **Versioning** | Keep all previous versions of a blob | `--enable-versioning true` |
| **Immutable storage** | WORM (Write Once Read Many) — can't modify or delete | Time-based retention or legal hold |
| **Infrastructure encryption** | Double encryption (SSE + infra-level) | `az storage account create --require-infrastructure-encryption` |

### BYOK (Bring Your Own Key)

By default, Azure encrypts storage with **Microsoft-managed keys**. BYOK lets you use your own key from Key Vault:

```bash
# Create key in Key Vault
az keyvault key create --vault-name my-kv -n storage-cmk --kty RSA --size 2048

# Configure storage to use customer-managed key
az storage account update -g rg-prod -n mysa \
  --encryption-key-source Microsoft.Keyvault \
  --encryption-key-vault https://my-kv.vault.azure.net \
  --encryption-key-name storage-cmk
```

**Requirements for BYOK**:
- Key Vault must have **soft delete** and **purge protection** enabled.
- Storage account must have a **managed identity** with **Key Vault Crypto Service Encryption User** role.

### Immutable storage

Two types:
- **Time-based retention**: can't delete until the retention period expires.
- **Legal hold**: can't delete until the legal hold is removed (no time limit).

```bash
# Set time-based immutability policy
az storage container immutability-policy create -g rg-prod \
  --account-name mysa -c compliance-data \
  --period 365 --allow-protected-append-writes true
```

---
## Summary

| Feature | Key exam fact |
|---------|---------------|
| **Entra RBAC** | Preferred over keys/SAS. Data-plane roles needed. |
| **SAS tokens** | Time-limited, scoped. Signed by account key. |
| **Stored access policies** | Revoke SAS without rotating keys. |
| **Soft delete** | Recover deleted blobs/containers. |
| **Immutable storage** | WORM. Time-based or legal hold. |
| **BYOK** | CMK in Key Vault. Requires soft delete + purge protection + MI. |
| **Infrastructure encryption** | Double encryption at infra level. |

**Next**: [Notebook 3 — Database Security](03_database_security.ipynb)